# Project 78 — JSON Extraction Dataset Builder
## Document → Schema → Extract → Validate → Export

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Define Extraction Schemas

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd
from pathlib import Path

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

# Define multiple target schemas
class PersonExtract(BaseModel):
    name: str
    role: str
    organization: str
    email: str = ""

class InvoiceExtract(BaseModel):
    invoice_id: str
    date: str
    vendor: str
    total: float
    items: list[str]

class EventExtract(BaseModel):
    title: str
    date: str
    time: str
    location: str
    attendees: list[str]

class ProductExtract(BaseModel):
    name: str
    sku: str
    price: float
    category: str
    features: list[str]

schemas = {
    "person": PersonExtract,
    "invoice": InvoiceExtract,
    "event": EventExtract,
    "product": ProductExtract,
}
print(f"Extraction schemas: {list(schemas.keys())}")

## Step 2 — Source Documents

In [ ]:
documents = [
    {"type": "person", "text": "Dr. Sarah Chen, Chief Data Officer at DataFlow Inc, "
     "has been leading the analytics transformation since 2022. Contact: sarah@dataflow.io"},
    {"type": "person", "text": "Meeting with John Smith (VP Engineering, TechCorp). "
     "Email: j.smith@techcorp.com. Discussed Q2 roadmap."},
    {"type": "invoice", "text": "Invoice #INV-2025-0042 from CloudServ Corp, dated Feb 1 2025. "
     "Items: API Gateway ($800), Storage addon ($200), Support ($500). Total: $1,500."},
    {"type": "invoice", "text": "Bill #B-9901 — DataPipe Solutions — March 10 2025. "
     "Professional Services (40hrs × $150 = $6000), License ($2000). Grand total: $8,000."},
    {"type": "event", "text": "Q1 All-Hands Meeting on March 25, 2025 at 2:00 PM in Conference Room A. "
     "Attendees: Alice, Bob, Carol, Dave, Eve. Agenda: roadmap review, team updates."},
    {"type": "product", "text": "ProMax Headphones (SKU: PMH-500), $79.99, Audio category. "
     "Features: active noise cancelling, 30hr battery, USB-C, Bluetooth 5.3, foldable design."},
]
print(f"Documents to process: {len(documents)}")

## Step 3 — Extract & Validate

In [ ]:
dataset = []
for doc in documents:
    schema_class = schemas[doc["type"]]
    extractor = llm.with_structured_output(schema_class)
    try:
        extracted = extractor.invoke(f"Extract structured data:\n{doc['text']}")
        extracted_dict = extracted.model_dump()
        valid = True
        error = ""
    except Exception as e:
        extracted_dict = {}
        valid = False
        error = str(e)[:80]

    dataset.append({
        "input": doc["text"],
        "type": doc["type"],
        "output": extracted_dict,
        "valid": valid,
        "error": error,
    })
    icon = "✓" if valid else "✗"
    print(f"  {icon} [{doc['type']}] {json.dumps(extracted_dict, default=str)[:100]}")

pass_rate = sum(1 for d in dataset if d["valid"]) / len(dataset)
print(f"\nExtraction pass rate: {pass_rate:.0%}")

## Step 4 — Generate Augmented Variants

In [ ]:
aug_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate 3 new documents similar to the example but with different data. "
     "Vary names, numbers, dates, and details. Return only the text, one per line."),
    ("human", "Example: {text}\n\nGenerate 3 variants:")
])
aug_chain = aug_prompt | llm | StrOutputParser()

augmented = []
for doc in documents[:4]:
    variants = aug_chain.invoke({"text": doc["text"]})
    for line in variants.strip().split("\n"):
        line = line.strip().lstrip("0123456789.-) ")
        if len(line) > 20:
            augmented.append({"type": doc["type"], "text": line})

print(f"Original: {len(documents)} | Augmented: {len(augmented)} | Total: {len(documents) + len(augmented)}")

# Export
Path("sample_data").mkdir(exist_ok=True)
with open("sample_data/extraction_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2, default=str)
print("✓ Saved to sample_data/extraction_dataset.json")

## What You Learned
- **Multi-schema extraction** from unstructured text
- **Pydantic validation** for output correctness
- **Data augmentation** to expand training sets
- **Training data export** for extraction fine-tuning